In [1]:
# ==========================
# 1. Configuração
# ==========================

import os
import re
import sqlite3
import platform
from datetime import datetime, date

import pandas as pd
import openpyxl
from openpyxl.utils import get_column_letter

ANO = 2026

if platform.system() == "Windows":
    DB_PATH = r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\00.DB\2026.db"
elif platform.system() == "Darwin":
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
else:
    raise RuntimeError("Sistema operativo não configurado.")

TABLE_NAME = "km_2026_V3"

In [3]:
# ==========================
# 1. Ler tabela da BD
# ==========================

import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

df_db = pd.read_sql_query(
    f"""
    SELECT *
    FROM {TABLE_NAME}
    ORDER BY
        data,
        transportador,
        trator,
        reboque
    """,
    conn,
)

conn.close()



In [5]:
df_db.head()

,id,chave,transportador,descricao,trator,reboque,data,dia,mes,km,portagens,valor_total,sub_divisao,tarifa_km,total,formula_subdivisao,criado_em
0,384,Florêncio & Silva|||2026-01-02,Florêncio & Silva,EXTRA,NaN,NaN,2026-01-02,2,1,200.0,NaN,220.0,0.00,0.39,220.00,=(D98-200)*0.39,2026-09-14 14:55:22
1,163,Florêncio & Silva|97-LF-45||2026-01-02,Florêncio & Silva,Carro 20 paletes (),97-LF-45,NaN,2026-01-02,2,1,154.0,NaN,243.5,75.46,0.49,318.96,=D26*0.49,2026-09-14 14:55:22
2,205,Florêncio & Silva|97-LF-46||2026-01-02,Florêncio & Silva,Carro 20 paletes (),97-LF-46,NaN,2026-01-02,2,1,124.0,NaN,235.0,60.76,0.49,295.76,=D38*0.49,2026-09-14 14:55:22
3,73,Florêncio & Silva|AR-59-PH|BB-70-RN|2026-01-02,Florêncio & Silva,NaN,AR-59-PH,BB-70-RN,2026-01-02,2,1,298.0,NaN,347.0,178.80,0.60,525.80,=D2*0.6,2026-09-14 14:55:22
4,293,Florêncio & Silva|AS-61-IG||2026-01-02,Florêncio & Silva,CAPILAR,AS-61-IG,NaN,2026-01-02,2,1,278.0,NaN,220.0,30.42,0.39,250.42,=(D74-200)*0.39,2026-09-14 14:55:22


In [6]:
# ==========================
# 1. Custo mensal
# ==========================

conn = sqlite3.connect(DB_PATH)

df_mensal_total = pd.read_sql_query(
    f"""
    SELECT
        strftime('%Y-%m', data) AS mes,
        ROUND(SUM(total), 2) AS custo_total
    FROM {TABLE_NAME}
    GROUP BY strftime('%Y-%m', data)
    ORDER BY mes
    """,
    conn,
)

df_mensal_prestador = pd.read_sql_query(
    f"""
    SELECT
        strftime('%Y-%m', data) AS mes,
        transportador AS prestador,
        ROUND(SUM(total), 2) AS custo_total
    FROM {TABLE_NAME}
    GROUP BY
        strftime('%Y-%m', data),
        transportador
    ORDER BY
        mes,
        custo_total DESC
    """,
    conn,
)

conn.close()

display(df_mensal_total)
display(df_mensal_prestador)

,mes,custo_total
0,2026-01,196255.80
1,2026-02,176733.33
2,2026-03,204008.62
3,2026-04,213679.09
4,2026-05,204253.96
5,2026-06,216512.69
6,2026-07,225563.67
7,2026-08,220898.02
8,2026-09,93690.26


,mes,prestador,custo_total
0,2026-01,Florêncio & Silva,116815.41
1,2026-01,Norte,50318.30
2,2026-01,Transaura,14908.93
3,2026-01,TJA,14213.16
4,2026-02,Florêncio & Silva,103617.40
5,2026-02,Norte,46436.10
6,2026-02,Transaura,13922.63
7,2026-02,TJA,12757.20
8,2026-03,Florêncio & Silva,130520.03
9,2026-03,Norte,50431.38


In [7]:
# ==========================
# 1. Ajuste de KM
# ==========================

df_db["km_ajustados"] = df_db["km"]

mask = (
    df_db["km"].ne(200)
    & df_db["km"].notna()
    & df_db["tarifa_km"].notna()
)

df_db.loc[mask, "km_ajustados"] = (
    df_db.loc[mask, "km"] * 0.95
)

df_db["custo_km_ajustado"] = 0.0

df_db.loc[mask, "custo_km_ajustado"] = (
    df_db.loc[mask, "km_ajustados"]
    * df_db.loc[mask, "tarifa_km"]
)

df_db[
    [
        "data",
        "transportador",
        "trator",
        "km",
        "km_ajustados",
        "tarifa_km",
        "custo_km_ajustado",
    ]
]

,data,transportador,trator,km,km_ajustados,tarifa_km,custo_km_ajustado
0,2026-01-02,Florêncio & Silva,NaN,200.0,200.00,0.39,0.0000
1,2026-01-02,Florêncio & Silva,97-LF-45,154.0,146.30,0.49,71.6870
2,2026-01-02,Florêncio & Silva,97-LF-46,124.0,117.80,0.49,57.7220
3,2026-01-02,Florêncio & Silva,AR-59-PH,298.0,283.10,0.60,169.8600
4,2026-01-02,Florêncio & Silva,AS-61-IG,278.0,264.10,0.39,102.9990
...,...,...,...,...,...,...,...
4949,2026-09-12,Norte,31-HP-00,110.0,104.50,0.39,40.7550
4950,2026-09-12,Norte,78-IU-76,68.0,64.60,0.39,25.1940
4951,2026-09-12,Norte,AO-00-DT,96.0,91.20,0.62,56.5440
4952,2026-09-12,Ramitrans,NaN,169.0,160.55,0.59,94.7245


In [8]:
# ==========================
# 1. Total antes e depois
# ==========================

df_db["km_ajustados"] = df_db["km"]

mask = (
    df_db["km"].ne(200)
    & df_db["km"].notna()
    & df_db["tarifa_km"].notna()
)

df_db.loc[mask, "km_ajustados"] = (
    df_db.loc[mask, "km"] * 0.95
)

df_db["custo_km_ajustado"] = 0.0

df_db.loc[mask, "custo_km_ajustado"] = (
    df_db.loc[mask, "km_ajustados"]
    * df_db.loc[mask, "tarifa_km"]
)

df_db["total_antes"] = df_db["total"]

df_db["total_depois"] = df_db["valor_total"]

df_db.loc[mask, "total_depois"] = (
    df_db.loc[mask, "valor_total"]
    + df_db.loc[mask, "custo_km_ajustado"]
)

df_db["diferenca"] = (
    df_db["total_depois"]
    - df_db["total_antes"]
)

df_db[
    [
        "data",
        "transportador",
        "trator",
        "km",
        "km_ajustados",
        "tarifa_km",
        "valor_total",
        "total_antes",
        "custo_km_ajustado",
        "total_depois",
        "diferenca",
    ]
]

,data,transportador,trator,km,km_ajustados,tarifa_km,valor_total,total_antes,custo_km_ajustado,total_depois,diferenca
0,2026-01-02,Florêncio & Silva,NaN,200.0,200.00,0.39,220.0,220.00,0.0000,220.0000,0.0000
1,2026-01-02,Florêncio & Silva,97-LF-45,154.0,146.30,0.49,243.5,318.96,71.6870,315.1870,-3.7730
2,2026-01-02,Florêncio & Silva,97-LF-46,124.0,117.80,0.49,235.0,295.76,57.7220,292.7220,-3.0380
3,2026-01-02,Florêncio & Silva,AR-59-PH,298.0,283.10,0.60,347.0,525.80,169.8600,516.8600,-8.9400
4,2026-01-02,Florêncio & Silva,AS-61-IG,278.0,264.10,0.39,220.0,250.42,102.9990,322.9990,72.5790
...,...,...,...,...,...,...,...,...,...,...,...
4949,2026-09-12,Norte,31-HP-00,110.0,104.50,0.39,234.0,276.90,40.7550,274.7550,-2.1450
4950,2026-09-12,Norte,78-IU-76,68.0,64.60,0.39,260.0,286.52,25.1940,285.1940,-1.3260
4951,2026-09-12,Norte,AO-00-DT,96.0,91.20,0.62,147.0,206.52,56.5440,203.5440,-2.9760
4952,2026-09-12,Ramitrans,NaN,169.0,160.55,0.59,230.0,347.05,94.7245,324.7245,-22.3255


In [10]:
# ==========================
# 1. Ganho mensal
# ==========================

df_ganho_mensal = (
    df_db
    .assign(
        mes=pd.to_datetime(
            df_db["data"],
            errors="coerce"
        ).dt.to_period("M").astype(str)
    )
    .groupby(
        "mes",
        as_index=False
    )["diferenca"]
    .sum()
    .rename(
        columns={
            "diferenca": "ganho"
        }
    )
)

df_ganho_mensal["ganho"] = (
    df_ganho_mensal["ganho"]
    .round(0)
    .astype(int)
)

df_ganho_mensal

,mes,ganho
0,2026-01,1388
1,2026-02,981
2,2026-03,2013
3,2026-04,1804
4,2026-05,2457
5,2026-06,2014
6,2026-07,3989
7,2026-08,1624
8,2026-09,584


In [11]:
# ==========================
# 1. Totais mensais
# ==========================

df_totais_mensais = (
    df_db
    .assign(
        mes=pd.to_datetime(
            df_db["data"],
            errors="coerce"
        ).dt.to_period("M").astype(str)
    )
    .groupby(
        "mes",
        as_index=False
    )
    .agg(
        total_antes=("total_antes", "sum"),
        total_depois=("total_depois", "sum"),
        diferenca=("diferenca", "sum"),
    )
)

df_totais_mensais[
    [
        "total_antes",
        "total_depois",
        "diferenca",
    ]
] = (
    df_totais_mensais[
        [
            "total_antes",
            "total_depois",
            "diferenca",
        ]
    ]
    .round(0)
    .astype(int)
)

df_totais_mensais

,mes,total_antes,total_depois,diferenca
0,2026-01,196256,197644,1388
1,2026-02,176733,177715,981
2,2026-03,204009,206022,2013
3,2026-04,213679,215483,1804
4,2026-05,204254,206711,2457
5,2026-06,216513,218526,2014
6,2026-07,225564,229552,3989
7,2026-08,220898,222522,1624
8,2026-09,93690,94274,584
